In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# Basic libraries
import os
import json
import cv2
import torch
import numpy as np
import pandas as pd

# PIL is used to convert OpenCV frames into images for the model
from PIL import Image

# Torchvision is needed to rebuild the ResNet18 model and define transforms
from torchvision import models, transforms

# display() is used at the end to show summary tables in the notebook
from IPython.display import display

In [11]:

# PATHS AND BASIC SETTINGS

BASE_DIR = "/content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization"
DEMO_DIR = os.path.join(BASE_DIR, "demo")
OUTPUT_DIR = DEMO_DIR

# Best checkpoint saved from the training notebook
best_ckpt_path = os.path.join(
    BASE_DIR,
    "experiments",
    "resnet18_baseline",
    "checkpoints",
    "resnet18_baseline_best.pt"
)

# Compiled demo video that will be annotated
DEMO_VIDEO_PATH = os.path.join(DEMO_DIR, "mie1076_ai_localisation_video_comp.mp4")

# Create output folder if needed
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Print paths so it is easy to confirm them before running
print("BASE_DIR:", BASE_DIR)
print("DEMO_DIR:", DEMO_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Checkpoint exists:", os.path.exists(best_ckpt_path), "->", best_ckpt_path)
print("Demo video exists:", os.path.exists(DEMO_VIDEO_PATH), "->", DEMO_VIDEO_PATH)

BASE_DIR: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization
DEMO_DIR: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/demo
OUTPUT_DIR: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/demo
Checkpoint exists: True -> /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/experiments/resnet18_baseline/checkpoints/resnet18_baseline_best.pt
Demo video exists: True -> /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/demo/mie1076_ai_localisation_video_comp.mp4


In [12]:
# MODEL / DEVICE / TRANSFORM SETUP

# Use GPU if Colab provides one, otherwise CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ResNet18 was trained with 224x224 images
IMG_SIZE = 224

# This transform matches the evaluation transform used in the training notebook
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def create_model(num_classes):
    """
    Rebuild the same ResNet18 architecture used during training.
    The final layer is replaced to match the number of classes.
    """
    model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = torch.nn.Linear(in_features, num_classes)
    return model

Using device: cpu


In [13]:
# DEMO CLIP GROUND-TRUTH LABELS
# Edit the timestamps and labels to match your compiled demo video.

demo_segments = [
    {"start_sec": 0.0,  "end_sec": 5.0,  "ground_truth": "ccu_lounge"},
    {"start_sec": 5.0,  "end_sec": 13.0,  "ground_truth": "floor3_elevator_landmark"},
    {"start_sec": 13.0,  "end_sec": 16.0, "ground_truth": "floor1_hallway"},
    {"start_sec": 16.0, "end_sec": 18.5, "ground_truth": "ccu_lounge"},
    {"start_sec": 18.5, "end_sec": 26.0, "ground_truth": "floor2_hallway"},
    {"start_sec": 26.0, "end_sec": 31.0, "ground_truth": "floor3_hallway"},
    {"start_sec": 31.0, "end_sec": 37.0, "ground_truth": "floor2_hallway"},
    {"start_sec": 37.0, "end_sec": 39.0, "ground_truth": "ccu_lounge"},
    {"start_sec": 39.0, "end_sec": 44.0, "ground_truth": "floor3_elevator_landmark"},
    {"start_sec": 44.0, "end_sec": 47.0, "ground_truth": "front_lobby"},
    {"start_sec": 47.0, "end_sec": 48.5, "ground_truth": "floor1_elevator_landmark"},
    {"start_sec": 48.5, "end_sec": 53.0, "ground_truth": "floor2_hallway"},
    {"start_sec": 53.0, "end_sec": 62.5, "ground_truth": "front_lobby"},
    {"start_sec": 62.5, "end_sec": 65.0, "ground_truth": "floor1_elevator_landmark"},
    {"start_sec": 65.0, "end_sec": 72.0, "ground_truth": "floor2_hallway"},
    {"start_sec": 72.0, "end_sec": 73.5, "ground_truth": "front_lobby"},
    {"start_sec": 73.5, "end_sec": 78.0, "ground_truth": "floor1_elevator_landmark"},
    {"start_sec": 78.0, "end_sec": 86.0, "ground_truth": "floor3_hallway"},
    {"start_sec": 86.0, "end_sec": 90.0, "ground_truth": "floor2_elevator_landmark"},
    {"start_sec": 90.0, "end_sec": 102.0, "ground_truth": "floor3_hallway"},
    {"start_sec": 102.0, "end_sec": 105.5, "ground_truth": "floor2_elevator_landmark"},
    {"start_sec": 105.5, "end_sec": 111.0, "ground_truth": "laundry"},
    {"start_sec": 111.0, "end_sec": 121.0, "ground_truth": "floor3_hallway"},
    {"start_sec": 121.0, "end_sec": 125.0, "ground_truth": "floor2_elevator_landmark"},
    {"start_sec": 125.0, "end_sec": 128.0, "ground_truth": "laundry"},
    {"start_sec": 128.0, "end_sec": 130.0, "ground_truth": "floor1_elevator_landmark"},
    {"start_sec": 130.0, "end_sec": 134.0, "ground_truth": "laundry"},
    {"start_sec": 134.0, "end_sec": 138.0, "ground_truth": "floor2_elevator_landmark"},
    {"start_sec": 138.0, "end_sec": 141.0, "ground_truth": "floor1_hallway"},
    {"start_sec": 141.0, "end_sec": 145.0, "ground_truth": "floor2_hallway"},
    {"start_sec": 145.0, "end_sec": 149.0, "ground_truth": "laundry"},
]

# To reduce compute, the notebook can predict every Nth frame
# and reuse the latest prediction on the frames in between.
predict_every_n_frames = 2

# Optional smoothing helps reduce flickering labels.
use_majority_smoothing = True
smoothing_window = 5

print("Number of labeled demo segments:", len(demo_segments))

Number of labeled demo segments: 31


In [14]:
# HELPER FUNCTIONS

def load_best_model_for_inference(best_ckpt_path, device):
    """
    Load the saved best model checkpoint and rebuild the model.

    Returns:
        model: loaded PyTorch model in eval mode
        loaded_class_names: class names saved inside the checkpoint
        checkpoint: full checkpoint dictionary
    """
    checkpoint = torch.load(best_ckpt_path, map_location=device)

    # Class names were saved when training finished
    loaded_class_names = checkpoint["class_names"]

    # Recreate the same model architecture and load weights
    model = create_model(num_classes=len(loaded_class_names)).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return model, loaded_class_names, checkpoint


def get_ground_truth_label(current_time_sec, segments):
    """
    Find the correct ground-truth label based on the current video time.
    If the current time falls inside one of the given clip segments,
    return that clip's ground-truth label.
    """
    for seg in segments:
        if seg["start_sec"] <= current_time_sec < seg["end_sec"]:
            return seg["ground_truth"]
    return None


def majority_vote(items):
    """
    Return the most common item in a list.
    Used for optional smoothing of predictions over recent frames.
    """
    if len(items) == 0:
        return None

    counts = {}
    for x in items:
        counts[x] = counts.get(x, 0) + 1

    return max(counts, key=counts.get)


def draw_label_box(frame, text_lines, top_left=(10, 10), box_width=760, line_height=32,
                   bg_color=(0, 0, 0), text_color=(255, 255, 255), thickness=2):
    """
    Draw a black box with multiple lines of text on top of a frame.
    This is used to show prediction, ground truth, confidence, and time.
    """
    x, y = top_left

    # Height depends on how many lines of text we want to show
    box_height = 10 + line_height * len(text_lines) + 10

    # Draw filled rectangle
    cv2.rectangle(frame, (x, y), (x + box_width, y + box_height), bg_color, -1)

    # Draw each line of text inside the rectangle
    for i, line in enumerate(text_lines):
        text_y = y + 30 + i * line_height
        cv2.putText(
            frame,
            line,
            (x + 10, text_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            text_color,
            thickness,
            cv2.LINE_AA
        )


def validate_segments(segments, class_names):
    """
    Check that the segment list is valid before processing the video.
    """
    for seg in segments:
        # Make sure required keys exist
        assert "start_sec" in seg and "end_sec" in seg and "ground_truth" in seg, \
            "Each segment must have start_sec, end_sec, and ground_truth"

        # Start must be before end
        assert seg["start_sec"] < seg["end_sec"], \
            "Each segment must satisfy start_sec < end_sec"

        # Ground truth must match one of the model's known classes
        assert seg["ground_truth"] in class_names, \
            f"Ground truth label {seg['ground_truth']} is not in class_names"

In [15]:

# MAIN FUNCTION

def annotate_demo_video_with_predictions(
    demo_video_path,
    demo_segments,
    best_ckpt_path,
    output_dir,
    predict_every_n_frames=1,
    use_majority_smoothing=False,
    smoothing_window=5
):
    """
    Read a demo video, run the model on it, overlay prediction and ground truth,
    save the annotated video, and save accuracy/results files.

    Inputs:
        demo_video_path: path to compiled demo video
        demo_segments: list of time segments with ground-truth labels
        best_ckpt_path: path to saved best model checkpoint
        output_dir: folder where outputs should be saved
        predict_every_n_frames: run model every N frames
        use_majority_smoothing: if True, smooth recent predictions
        smoothing_window: number of recent predictions used for smoothing

    Outputs saved:
        - annotated video
        - frame-level CSV
        - segment-level CSV
        - summary JSON
    """

    # Basic checks before starting
    assert os.path.exists(best_ckpt_path), f"Checkpoint not found: {best_ckpt_path}"
    assert os.path.exists(demo_video_path), f"Demo video not found: {demo_video_path}"
    assert predict_every_n_frames >= 1, "predict_every_n_frames must be at least 1"

    # Make sure output folder exists
    os.makedirs(output_dir, exist_ok=True)

    # Load model from checkpoint
    model, loaded_class_names, checkpoint = load_best_model_for_inference(best_ckpt_path, DEVICE)

    # Check that segment labels are valid
    validate_segments(demo_segments, loaded_class_names)

    # Open the input video
    cap = cv2.VideoCapture(demo_video_path)
    assert cap.isOpened(), f"Could not open video: {demo_video_path}"

    # Read basic video information
    fps = cap.get(cv2.CAP_PROP_FPS)
    assert fps > 0, "Could not read FPS from video."

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = total_frames / fps

    # Build output file names
    video_name = os.path.splitext(os.path.basename(demo_video_path))[0]
    output_video_path = os.path.join(output_dir, f"mie1076_ai_localisation_demo.mp4")
    frame_results_csv = os.path.join(output_dir, f"{video_name}_frame_results.csv")
    segment_results_csv = os.path.join(output_dir, f"{video_name}_segment_results.csv")
    summary_json_path = os.path.join(output_dir, f"{video_name}_summary.json")

    # Create video writer for saving the annotated video
    # mp4v usually works well in Colab for .mp4 output
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    assert writer.isOpened(), f"Could not create output video: {output_video_path}"

    # Variables used while looping through frames
    frame_idx = 0
    last_pred_label = None
    last_confidence = None
    prediction_history = []

    # Store frame-by-frame results here
    frame_rows = []

    while True:
        ret, frame_bgr = cap.read()

        # Stop when video ends
        if not ret:
            break

        # Current frame time in seconds
        current_time_sec = frame_idx / fps

        # Find the correct label for the current clip segment
        gt_label = get_ground_truth_label(current_time_sec, demo_segments)

        # Only run the model every N frames if requested
        should_predict = (frame_idx % predict_every_n_frames == 0)

        if should_predict:
            # Convert OpenCV frame (BGR) to PIL image (RGB)
            pil_img = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))

            # Apply the same evaluation transform used during model evaluation
            input_tensor = eval_transform(pil_img).unsqueeze(0).to(DEVICE)

            # Run inference
            with torch.no_grad():
                outputs = model(input_tensor)

                # Convert logits to probabilities
                probs = torch.softmax(outputs, dim=1)

                # Predicted class index
                pred_idx = torch.argmax(probs, dim=1).item()

                # Confidence for the predicted class
                pred_conf = probs[0, pred_idx].item()

            raw_pred_label = loaded_class_names[pred_idx]

            # Optional smoothing across recent predictions
            if use_majority_smoothing:
                prediction_history.append(raw_pred_label)

                # Keep only the most recent N predictions
                if len(prediction_history) > smoothing_window:
                    prediction_history.pop(0)

                pred_label = majority_vote(prediction_history)
            else:
                pred_label = raw_pred_label

            # Save latest prediction so skipped frames can reuse it
            last_pred_label = pred_label
            last_confidence = pred_conf

        # If this frame was skipped, reuse the most recent prediction
        pred_label = last_pred_label
        pred_conf = last_confidence

        # Frame is correct only if we have a ground truth and prediction matches it
        is_correct = (gt_label is not None and pred_label == gt_label)

        # Make prediction text green if correct, red if wrong
        pred_color = (0, 255, 0) if is_correct else (0, 0, 255)

        # Text lines to display
        line_1 = f"time: {current_time_sec:.2f}s"
        line_2 = f"prediction: {pred_label if pred_label is not None else 'N/A'}"
        line_3 = f"ground_truth: {gt_label if gt_label is not None else 'N/A'}"
        line_4 = f"confidence: {pred_conf:.3f}" if pred_conf is not None else "confidence: N/A"
        line_5 = f"correct: {is_correct}"

        # Draw the main label box
        draw_label_box(
            frame_bgr,
            [line_1, line_2, line_3, line_4, line_5],
            top_left=(10, 10),
            box_width=760,
            line_height=32,
            bg_color=(0, 0, 0),
            text_color=(255, 255, 255),
            thickness=2
        )

        # Draw the prediction line again in green/red so it stands out more
        cv2.putText(
            frame_bgr,
            f"prediction: {pred_label if pred_label is not None else 'N/A'}",
            (20, 72),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            pred_color,
            2,
            cv2.LINE_AA
        )

        # Save the annotated frame into the output video
        writer.write(frame_bgr)

        # Save one row for later analysis
        frame_rows.append({
            "frame_idx": frame_idx,
            "time_sec": current_time_sec,
            "ground_truth": gt_label,
            "prediction": pred_label,
            "confidence": pred_conf,
            "correct": is_correct
        })

        frame_idx += 1

    # Release input and output video objects
    cap.release()
    writer.release()

    # Convert frame results to DataFrame and save
    frame_df = pd.DataFrame(frame_rows)
    frame_df.to_csv(frame_results_csv, index=False)

    # Keep only rows where a ground truth exists
    eval_df = frame_df[frame_df["ground_truth"].notna()].copy()

    # Overall frame-level accuracy
    overall_accuracy = float(eval_df["correct"].mean()) if len(eval_df) > 0 else None

    # Build per-segment results
    segment_rows = []

    for i, seg in enumerate(demo_segments):
        seg_df = eval_df[
            (eval_df["time_sec"] >= seg["start_sec"]) &
            (eval_df["time_sec"] < seg["end_sec"])
        ].copy()

        segment_acc = float(seg_df["correct"].mean()) if len(seg_df) > 0 else None

        majority_pred = None
        if len(seg_df) > 0:
            preds = seg_df["prediction"].dropna().tolist()
            majority_pred = majority_vote(preds) if len(preds) > 0 else None

        segment_rows.append({
            "segment_id": i,
            "start_sec": seg["start_sec"],
            "end_sec": seg["end_sec"],
            "ground_truth": seg["ground_truth"],
            "num_frames": len(seg_df),
            "segment_accuracy": segment_acc,
            "majority_prediction": majority_pred
        })

    # Save segment-level table
    segment_df = pd.DataFrame(segment_rows)
    segment_df.to_csv(segment_results_csv, index=False)

    # Save summary information
    summary = {
        "video_path": demo_video_path,
        "output_video_path": output_video_path,
        "frame_results_csv": frame_results_csv,
        "segment_results_csv": segment_results_csv,
        "num_total_frames": int(total_frames),
        "fps": float(fps),
        "duration_sec": float(duration_sec),
        "predict_every_n_frames": int(predict_every_n_frames),
        "use_majority_smoothing": bool(use_majority_smoothing),
        "smoothing_window": int(smoothing_window),
        "overall_frame_accuracy": overall_accuracy,
        "num_segments": len(demo_segments)
    }

    with open(summary_json_path, "w") as f:
        json.dump(summary, f, indent=2)

    # Print saved output locations
    print("Saved annotated video to:", output_video_path)
    print("Saved frame-level results to:", frame_results_csv)
    print("Saved segment-level results to:", segment_results_csv)
    print("Saved summary to:", summary_json_path)

    if overall_accuracy is not None:
        print(f"Overall frame accuracy: {overall_accuracy:.4f}")

    # Show the segment summary table inside the notebook
    display(segment_df)

    return {
        "output_video_path": output_video_path,
        "frame_results_csv": frame_results_csv,
        "segment_results_csv": segment_results_csv,
        "summary_json_path": summary_json_path,
        "frame_df": frame_df,
        "segment_df": segment_df
    }

In [16]:
# RUN THE DEMO VIDEO ANNOTATION


demo_result = annotate_demo_video_with_predictions(
    demo_video_path=DEMO_VIDEO_PATH,
    demo_segments=demo_segments,
    best_ckpt_path=best_ckpt_path,
    output_dir=OUTPUT_DIR,
    predict_every_n_frames=predict_every_n_frames,
    use_majority_smoothing=use_majority_smoothing,
    smoothing_window=smoothing_window
)

Saved annotated video to: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/demo/mie1076_ai_localisation_demo.mp4
Saved frame-level results to: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/demo/mie1076_ai_localisation_video_comp_frame_results.csv
Saved segment-level results to: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/demo/mie1076_ai_localisation_video_comp_segment_results.csv
Saved summary to: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/demo/mie1076_ai_localisation_video_comp_summary.json
Overall frame accuracy: 0.6886


,segment_id,start_sec,end_sec,ground_truth,num_frames,segment_accuracy,majority_prediction
0,0,0.0,5.0,ccu_lounge,150,0.786667,ccu_lounge
1,1,5.0,13.0,floor3_elevator_landmark,240,0.841667,floor3_elevator_landmark
2,2,13.0,16.0,floor1_hallway,90,0.755556,floor1_hallway
3,3,16.0,18.5,ccu_lounge,75,0.893333,ccu_lounge
4,4,18.5,26.0,floor2_hallway,225,0.986667,floor2_hallway
5,5,26.0,31.0,floor3_hallway,150,0.480000,floor3_hallway
6,6,31.0,37.0,floor2_hallway,180,0.888889,floor2_hallway
7,7,37.0,39.0,ccu_lounge,60,0.966667,ccu_lounge
8,8,39.0,44.0,floor3_elevator_landmark,150,0.933333,floor3_elevator_landmark
9,9,44.0,47.0,front_lobby,90,0.822222,front_lobby


In [17]:
# QUICK RESULT PREVIEW

# Show the first few frame-level rows
display(demo_result["frame_df"].head())

# Show the segment-level summary
display(demo_result["segment_df"])

,frame_idx,time_sec,ground_truth,prediction,confidence,correct
0,0,0.000000,ccu_lounge,floor2_hallway,0.957786,False
1,1,0.033333,ccu_lounge,floor2_hallway,0.957786,False
2,2,0.066667,ccu_lounge,floor2_hallway,0.957786,False
3,3,0.100000,ccu_lounge,floor2_hallway,0.957786,False
4,4,0.133333,ccu_lounge,floor2_hallway,0.957786,False


,segment_id,start_sec,end_sec,ground_truth,num_frames,segment_accuracy,majority_prediction
0,0,0.0,5.0,ccu_lounge,150,0.786667,ccu_lounge
1,1,5.0,13.0,floor3_elevator_landmark,240,0.841667,floor3_elevator_landmark
2,2,13.0,16.0,floor1_hallway,90,0.755556,floor1_hallway
3,3,16.0,18.5,ccu_lounge,75,0.893333,ccu_lounge
4,4,18.5,26.0,floor2_hallway,225,0.986667,floor2_hallway
5,5,26.0,31.0,floor3_hallway,150,0.480000,floor3_hallway
6,6,31.0,37.0,floor2_hallway,180,0.888889,floor2_hallway
7,7,37.0,39.0,ccu_lounge,60,0.966667,ccu_lounge
8,8,39.0,44.0,floor3_elevator_landmark,150,0.933333,floor3_elevator_landmark
9,9,44.0,47.0,front_lobby,90,0.822222,front_lobby
